# Did the One-Child Policy Actually Work?
### A Synthetic Control Analysis of China's 1979 Fertility Policy

**Author:** Srisha Raj 
**Data:** World Bank World Development Indicators  
**Method:** Synthetic Control Method (Abadie, Diamond & Hainmueller, 2010)  
**Companion report:** `china_onechild_scm_report.docx`

---

## Research Question

China's fertility rate was already falling sharply before 1979, from 6.1 births per woman in 1960 to 2.8 by the time the One-Child Policy launched. **How much of that decline would have happened anyway, without any policy?**

To answer this, we need a credible estimate of what China's fertility trajectory would have looked like without the policy. 

### Notebook Structure
1. Setup & Data Loading
2. Exploratory Data Analysis
3. Motivating Analysis: Difference-in-Differences (DiD)
4. Main Analysis: Synthetic Control Method (SCM)
5. Results & Interpretation


## 1. Setup & Data Loading

In [9]:
# Install required library if not already present
# pysyncon implements the Abadie et al. (2010) Synthetic Control Method
try:
    import pysyncon
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pysyncon', '-q',
                           '--break-system-packages'])
    import pysyncon


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
import seaborn as sns
from pysyncon import Dataprep, Synth

# Plot styling
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})


ModuleNotFoundError: No module named 'seaborn'

### 1.1 Load Raw Data

Four World Bank indicators are used:

| Indicator | Code | Role |
|---|---|---|
| Total Fertility Rate | SP.DYN.TFRT.IN | Outcome variable |
| GDP (current USD) | NY.GDP.MKTP.CD | Predictor — economic development |
| Urban population (% of total) | SP.URB.TOTL.IN.ZS | Predictor — structural modernization |
| Under-5 mortality rate | SH.DYN.MORT | Predictor — child survival dynamics |

**Note on country names:** The World Bank uses `Korea, Rep.`, `Viet Nam`, and `Turkiye` — these are standardized below.


In [ ]:
# ── File paths ────────────────────────────────────────────────────────────
# All raw data lives in the data/ subfolder.
# Files are renamed from World Bank download names for portability.
DATA = {
    'fertility' : 'data/fertility_rate.csv',
    'gdp'       : 'data/gdp.csv',
    'urban'     : 'data/urban_population_pct.csv',
    'mortality' : 'data/child_mortality.csv',
}

# ── Country configuration ─────────────────────────────────────────────────
# Vietnam excluded: had its own government fertility campaigns from late 1970s,
# which would contaminate the counterfactual.
DONOR_COUNTRIES = [
    'Thailand', 'Turkey', 'Pakistan', 'Philippines',
    'Sri Lanka', 'Morocco', 'Tunisia', 'South Korea',
    'Brazil', 'Mexico'
]
ALL_COUNTRIES = ['China'] + DONOR_COUNTRIES

# World Bank uses non-standard names for some countries
NAME_MAP = {
    'Korea, Rep.'  : 'South Korea',
    'Viet Nam'     : 'Vietnam',
    'Turkiye'      : 'Turkey',
}

STR_YEARS = [str(y) for y in range(1960, 2016)]

def load_wb(path, indicator_name=None):
    """Load a World Bank wide-format CSV and return a long-format DataFrame."""
    df = pd.read_csv(path, skiprows=4)
    if indicator_name:
        df = df[df['Indicator Name'] == indicator_name]
    df = df[['Country Name'] + STR_YEARS].copy()
    df['Country Name'] = df['Country Name'].replace(NAME_MAP)
    df = df[df['Country Name'].isin(ALL_COUNTRIES)]
    df = df.melt(id_vars='Country Name', var_name='Year', value_name='value')
    df['Year'] = df['Year'].astype(int)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df


In [ ]:
fert  = load_wb(DATA['fertility']).rename(columns={'value': 'Fertility'})
gdp   = load_wb(DATA['gdp']).rename(columns={'value': 'GDP'})
urban = load_wb(DATA['urban'],
                indicator_name='Urban population (% of total population)'
               ).rename(columns={'value': 'Urban_Pct'})
mort  = load_wb(DATA['mortality']).rename(columns={'value': 'Mortality'})

# Merge all indicators; log-transform GDP (right-skewed across donor pool)
df = (fert
      .merge(gdp,   on=['Country Name', 'Year'])
      .merge(urban, on=['Country Name', 'Year'])
      .merge(mort,  on=['Country Name', 'Year']))

df['log_GDP'] = np.log(df['GDP'])
df = df.drop(columns='GDP')

# Restrict to 1969–2015
# Lower bound: 1969 is the first year China has complete mortality data
# Upper bound: 2015 is used to avoid post-2015 data sparsity in some donors
df = df[(df['Year'] >= 1969) & (df['Year'] <= 2015)]

print(f"Shape: {df.shape}")
print(f"Countries: {sorted(df['Country Name'].unique())}")
print(f"Missing values:\n{df.isnull().sum()}")


---
## 2. Exploratory Data Analysis

Before running any causal model, we need to understand the data. Three questions:
1. What did China's fertility trajectory look like, and when did it start falling?
2. How do the donor countries compare to China pre-1979?
3. Do any donors look like plausible comparators?


In [ ]:
# ── Fig 1: China's full fertility trajectory ──────────────────────────────
china = df[df['Country Name'] == 'China'].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(china['Year'], china['Fertility'], color='#1B4F72', linewidth=2.5, label='China TFR')
ax.axvline(1979, color='crimson', linestyle='--', linewidth=1.5, label='One-Child Policy (1979)')
ax.set_title("China: Total Fertility Rate (1969–2015)", fontsize=13, fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Births per Woman")
ax.legend()
plt.tight_layout()
plt.show()

print(f"China TFR in 1969: {china[china['Year']==1969]['Fertility'].values[0]:.2f}")
print(f"China TFR in 1979: {china[china['Year']==1979]['Fertility'].values[0]:.2f}")
print(f"China TFR in 2000: {china[china['Year']==2000]['Fertility'].values[0]:.2f}")


In [ ]:
# ── Fig 2: All countries — full picture ───────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

for country in DONOR_COUNTRIES:
    d = df[df['Country Name'] == country]
    ax.plot(d['Year'], d['Fertility'], color='#AAAAAA', linewidth=1, alpha=0.7)
    ax.text(d['Year'].max() + 0.3, d[d['Year'] == d['Year'].max()]['Fertility'].values[0],
            country, fontsize=7, color='#888888', va='center')

# China on top
ax.plot(china['Year'], china['Fertility'], color='#1B4F72', linewidth=2.5, label='China', zorder=5)
ax.axvline(1979, color='crimson', linestyle='--', linewidth=1.5, label='One-Child Policy (1979)')

ax.set_title("Fertility Rate: China vs. Donor Pool (1969–2015)", fontsize=13, fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Births per Woman")
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# ── Fig 3: Pre-treatment period only (1969–1978) ─────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

pre = df[df['Year'] <= 1978]
for country in DONOR_COUNTRIES:
    d = pre[pre['Country Name'] == country]
    ax.plot(d['Year'], d['Fertility'], color='#AAAAAA', linewidth=1.2, alpha=0.8)

ax.plot(pre[pre['Country Name']=='China']['Year'],
        pre[pre['Country Name']=='China']['Fertility'],
        color='#1B4F72', linewidth=2.5, label='China')

ax.set_title("Pre-Treatment Fertility (1969–1978): Is a match possible?", fontsize=12, fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Births per Woman")
ax.legend()
plt.tight_layout()
plt.show()


---
## 3. Motivating Analysis: Difference-in-Differences

Before the main synthetic control analysis, we run a simple Difference-in-Differences (DiD) regression using South Korea as a single comparison country.

**DiD logic:** If China and South Korea were on parallel fertility trajectories before 1979, any divergence after 1979 can be attributed to the One-Child Policy.

**Limitation (why we need SCM):** DiD with a single comparison country is fragile — it assumes that one country is a perfect counterfactual for China. The synthetic control method relaxes this by constructing an *optimal weighted blend* of multiple countries.


In [ ]:
try:
    import statsmodels.formula.api as smf
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'statsmodels', '-q',
                           '--break-system-packages'])
    import statsmodels.formula.api as smf

# Build DiD dataset: China vs South Korea
did_df = df[df['Country Name'].isin(['China', 'South Korea'])].copy()
did_df['log_Fertility'] = np.log(did_df['Fertility'])
did_df['Treatment'] = (did_df['Country Name'] == 'China').astype(int)
did_df['Post'] = (did_df['Year'] >= 1979).astype(int)
did_df['DiD'] = did_df['Treatment'] * did_df['Post']

model = smf.ols('log_Fertility ~ Treatment + Post + DiD', data=did_df).fit(
    cov_type='HC3'  # heteroskedasticity-robust standard errors
)
print(model.summary())


In [ ]:
# ── Fig 4: Parallel trends check (pre-1979) ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for country, color, lw in [('China', '#1B4F72', 2.5), ('South Korea', '#E67E22', 2)]:
    d = did_df[did_df['Country Name'] == country]
    ax.plot(d['Year'], d['Fertility'], color=color, linewidth=lw, label=country)

ax.axvline(1979, color='crimson', linestyle='--', linewidth=1.5, label='One-Child Policy (1979)')
ax.set_title("DiD: China vs. South Korea Fertility (1969–2015)", fontsize=12, fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Births per Woman")
ax.legend()
plt.tight_layout()
plt.show()

print("\nNote: Pre-1979 trends are not perfectly parallel — South Korea's")
print("fertility was higher and falling faster than China's.")
print("This motivates using Synthetic Control, which constructs a better-matched counterfactual.")


---
## 4. Main Analysis: Synthetic Control Method (SCM)

### What is SCM?

The Synthetic Control Method (Abadie et al., 2010) constructs a counterfactual "Synthetic China" as a *weighted combination* of donor countries. The weights are chosen to minimize the difference between real China and the synthetic unit across:
- Pre-treatment fertility trajectory (1969–1978)
- Four pre-treatment predictor variables: log GDP, urbanization, child mortality, mean fertility

The key insight: instead of relying on one imperfect comparator (as in DiD), SCM finds the *optimal blend* that best replicates China's pre-policy characteristics.

After 1979, we observe how real China diverges from Synthetic China — that gap is our estimate of the policy's effect.


In [ ]:
# ── Build Dataprep object ─────────────────────────────────────────────────
PRE_YEARS  = list(range(1969, 1979))   # optimization window: 1969–1978
ALL_YEARS  = list(range(1969, 2016))   # plot window: 1969–2015

dataprep = Dataprep(
    foo=df,
    predictors=['log_GDP', 'Urban_Pct', 'Mortality'],
    predictors_op='mean',
    time_predictors_prior=PRE_YEARS,
    special_predictors=[
        ('Fertility', PRE_YEARS, 'mean'),   # match on mean pre-treatment fertility
    ],
    dependent='Fertility',
    unit_variable='Country Name',
    time_variable='Year',
    treatment_identifier='China',
    controls_identifier=DONOR_COUNTRIES,
    time_optimize_ssr=PRE_YEARS,
)

synth = Synth()
synth.fit(dataprep)
print("Optimization complete.")


In [ ]:
# ── Donor weights ─────────────────────────────────────────────────────────
weights_df = pd.DataFrame({
    'Country': DONOR_COUNTRIES,
    'Weight' : synth.W.flatten().round(4)
}).sort_values('Weight', ascending=False)

print("=== Synthetic Control Donor Weights ===")
print(weights_df[weights_df['Weight'] > 0.001].to_string(index=False))
print("\nNote: SCM assigns zero weight to countries that don't contribute")
print("to matching China's pre-treatment trajectory.")


In [ ]:
# ── Build synthetic series manually ───────────────────────────────────────
outcome_wide = df.pivot(index='Year', columns='Country Name', values='Fertility')
synthetic = (outcome_wide[DONOR_COUNTRIES].values @ synth.W).flatten()
china_actual = outcome_wide['China'].values
years_arr = outcome_wide.index.values

# Pre-treatment RMSPE (fit quality diagnostic)
pre_mask  = years_arr < 1979
post_mask = years_arr >= 1979

rmspe_pre  = np.sqrt(np.mean((china_actual[pre_mask]  - synthetic[pre_mask])**2))
rmspe_post = np.sqrt(np.mean((china_actual[post_mask] - synthetic[post_mask])**2))
gap        = china_actual - synthetic

print(f"Pre-treatment RMSPE  (fit quality): {rmspe_pre:.4f}")
print(f"Post-treatment RMSPE (effect size): {rmspe_post:.4f}")
print(f"Post/Pre RMSPE ratio:               {rmspe_post/rmspe_pre:.2f}")
print(f"\nAvg post-treatment gap (China − Synthetic): {gap[post_mask].mean():.4f} TFR")
print("(Negative = China below synthetic = policy suppressed fertility)")
print("(Near zero = policy effect indistinguishable from counterfactual trend)")


---
## 5. Results & Interpretation


In [ ]:
# ── Fig 5: Main SCM result ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Actual vs Synthetic
ax = axes[0]
ax.plot(years_arr, china_actual, color='#1B4F72', linewidth=2.5, label='China (Actual)')
ax.plot(years_arr, synthetic,    color='#E67E22', linewidth=2, linestyle='--',
        label='Synthetic China')
ax.axvline(1979, color='crimson', linestyle=':', linewidth=1.5, label='Policy Start (1979)')
ax.set_title("China vs. Synthetic China\nTotal Fertility Rate", fontsize=12, fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Births per Woman")
ax.legend()

# Panel B: Gap
ax = axes[1]
ax.plot(years_arr, gap, color='#2C3E50', linewidth=2)
ax.axvline(1979, color='crimson', linestyle=':', linewidth=1.5, label='Policy Start (1979)')
ax.axhline(0, color='gray', linewidth=0.8)
ax.fill_between(years_arr, gap, 0,
                where=(years_arr >= 1979),
                alpha=0.15, color='crimson',
                label='Post-treatment period')
ax.set_title("Gap: China − Synthetic China\n(Estimated Policy Effect)", fontsize=12, fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Gap in TFR (births per woman)")
ax.legend()

plt.suptitle("Synthetic Control Results: One-Child Policy Analysis",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/scm_main_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 6: Pre-treatment fit detail ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

pre_years   = years_arr[pre_mask]
ax.plot(pre_years, china_actual[pre_mask], color='#1B4F72', linewidth=2.5,
        marker='o', markersize=4, label='China (Actual)')
ax.plot(pre_years, synthetic[pre_mask],    color='#E67E22', linewidth=2,
        linestyle='--', marker='s', markersize=4, label='Synthetic China')

ax.set_title(f"Pre-Treatment Fit (1969–1978)\nRMSPE = {rmspe_pre:.3f}",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Births per Woman")
ax.legend()
plt.tight_layout()
plt.savefig('figures/scm_pretreatment_fit.png', dpi=150, bbox_inches='tight')
plt.show()


### Interpretation

**Donor weights:** The optimizer assigned meaningful weight to only three countries — Thailand (76.8%), South Korea (21.4%), and Turkey (1.8%). The remaining seven received zero weight. Synthetic China is essentially a blend of rapid-development-era Thailand and South Korea: two economies that underwent steep fertility declines in the same period, without comparable coercive policies.

**Pre-treatment fit:** The RMSPE of 0.596 represents a reasonable but imperfect match. The synthetic unit tracks China's general downward trend but sits somewhat above real China in the early 1970s — a reflection of China's unusually sharp early decline, which the donor pool partially struggles to replicate.

**Post-treatment gap:** After 1979, Synthetic China and real China decline at nearly identical rates. The average post-treatment gap is +0.054 TFR — meaning real China's fertility was marginally *higher* than the synthetic counterfactual, not lower. There is no persistent downward divergence.

**What this means:** If the One-Child Policy had a strong suppressive effect, we would expect real China to fall well below Synthetic China after 1979 and stay there. That pattern does not appear. The countries that most resemble China demographically — Thailand and South Korea — declined at nearly the same rate without a one-child policy.

**Two defensible readings:**
1. **Structural dominance:** The fertility transition was largely driven by economic modernization forces (urbanization, income growth, child survival) that would have produced similar declines regardless of the policy.
2. **Donor pool limitation:** No combination of these ten countries can cleanly replicate China's unique pre-1979 trajectory, meaning a real policy effect may be obscured by noise in the counterfactual.

Both are likely true to some degree. National-level analysis of a single country is always underpowered for detecting effects layered on top of strong pre-existing trends.


---
## Limitations & Next Steps

**Limitations:**
- **Unit of analysis:** Using China as a single unit aggregates enormous internal variation. The One-Child Policy was implemented unevenly across provinces, urban/rural settings, and ethnic minority populations.
- **Pre-treatment fit:** An RMSPE of 0.596 is acceptable but not tight. A tighter fit would give more confidence in the counterfactual.
- **No placebo test:** The standard robustness check (running the same procedure for each donor country as a falsification test) was not conducted here. This would help assess whether the China result is unusually small relative to the placebo distribution.
- **Single outcome:** TFR captures births per woman but not the policy's effects on sex ratios, forced procedures, or long-run family structure.

**Next Steps:**
- **Province-level analysis:** Compare provinces with early, strict implementation to those with delayed or weak enforcement — variation within China avoids the cross-country donor pool problem entirely.
- **Alternative outcomes:** Sex ratio at birth, female labor force participation, household size — these may show sharper, more clearly policy-attributable effects.
- **Placebo testing:** Implement the Abadie et al. permutation-based inference procedure to assess statistical significance of the post-treatment gap.

---
*Srisha Raj · UC Berkeley · 2025*  
*Data: World Bank WDI · Code: github.com/srisha-raj/china-onechild-policy-scm*
